# Gemini Generation Pipeline

This notebook generates short representative reviews from sampled IMDb reviews and scores them with the trained regression judge. It preserves the original project flow while using repository-relative paths and environment variables for local reruns.

**Required environment variable:** `GEMINI_API_KEY`

**Optional environment variables:** `JUDGE_MODEL_PATH`, `GEMINI_MODEL_NAME`, `GEMINI_MODEL_LABEL`

**Outputs:** `outputs/results_gemini_3_0_flash.xlsx`, `outputs/summary_statistics.xlsx`


In [ ]:
from pathlib import Path

import os
import time

import google.generativeai as genai
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

PROJECT_ROOT = next(
    (path.resolve() for path in (Path.cwd(), Path.cwd().parent) if (path / "data" / "imdb_sup.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate data/imdb_sup.csv. Run the notebook from the repository root or notebooks/ directory."
    )

DATA_PATH = PROJECT_ROOT / 'data' / 'imdb_sup.csv'
DEFAULT_MODEL_PATH = PROJECT_ROOT / 'artifacts' / 'judge_regression_model'
MODEL_PATH = Path(os.getenv('JUDGE_MODEL_PATH', str(DEFAULT_MODEL_PATH))).expanduser()
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUTS_DIR / 'results_gemini_3_0_flash.xlsx'
SUMMARY_FILE = OUTPUTS_DIR / 'summary_statistics.xlsx'

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise EnvironmentError('Set GEMINI_API_KEY before running this notebook.')

MODEL_NAME = os.getenv('GEMINI_MODEL_NAME', 'models/gemini-3-flash-preview')
RESULT_MODEL_LABEL = os.getenv('GEMINI_MODEL_LABEL', 'Gemini 3.0 Flash')
NUM_ITERATIONS = 10
TARGET_RATINGS = [1, 2, 3, 4, 7, 8, 9, 10]

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Judge model not found at {MODEL_PATH}. Train it first or set JUDGE_MODEL_PATH.'
    )

genai.configure(api_key=GEMINI_API_KEY)

print('Loading judge model and dataset...')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded_model = DistilBertForSequenceClassification.from_pretrained(str(MODEL_PATH)).to(device)
loaded_tokenizer = DistilBertTokenizerFast.from_pretrained(str(MODEL_PATH))
loaded_model.eval()
df = pd.read_csv(DATA_PATH)

def load_existing_results(file_path):
    if file_path.exists():
        try:
            return pd.read_excel(file_path).to_dict('records')
        except Exception:
            return []
    return []

results = load_existing_results(OUTPUT_FILE)
print(f'Found {len(results)} existing records. Resuming from where we stopped...')


In [ ]:
def predict_rating(text):
    if not text or text == 'ERROR':
        return 0, 0

    inputs = loaded_tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=512,
    ).to(device)

    with torch.no_grad():
        outputs = loaded_model(**inputs)

    logits = outputs.logits
    if logits.shape[1] > 1:
        pred_label = torch.argmax(logits, dim=1).item()
        label_to_rating = {0: 1, 1: 2, 2: 3, 3: 4, 4: 7, 5: 8, 6: 9, 7: 10}
        return float(pred_label), label_to_rating.get(pred_label, 1)

    raw_val = logits.item()
    valid = np.array([1, 2, 3, 4, 7, 8, 9, 10])
    return raw_val, int(valid[(np.abs(valid - raw_val)).argmin()])

def call_gemini_smart(prompt):
    model = genai.GenerativeModel(MODEL_NAME)
    retry_count = 0
    while True:
        try:
            response = model.generate_content(prompt, request_options={'timeout': 60})
            return response.text.strip(), 'SUCCESS'
        except Exception as exc:
            err_msg = str(exc).lower()
            if '429' in err_msg:
                retry_count += 1
                if retry_count >= 2:
                    print('\nDaily request limit appears to be reached. Stopping this run.')
                    return None, 'DAILY_LIMIT'
                print(f'\nRate limit reached. Waiting 60 seconds before retry {retry_count}/2...')
                time.sleep(60)
            else:
                print(f'\nAPI error: {exc}. Retrying in 10 seconds...')
                time.sleep(10)

stop_execution = False
print(f'Starting or resuming experiment for: {MODEL_NAME}')

for rating in TARGET_RATINGS:
    if stop_execution:
        break

    rating_df = df[df['Rating'] == rating]
    for prompt_type in ['Basic', 'Persona']:
        if stop_execution:
            break

        done_count = sum(
            1
            for row in results
            if row['Target_Rating'] == rating and row['Prompt_Type'] == prompt_type
        )
        needed = NUM_ITERATIONS - done_count
        if needed <= 0:
            continue

        print(f'\n>>> Rating {rating} | {prompt_type} | Progress: {done_count}/{NUM_ITERATIONS}')

        for _ in tqdm(range(needed)):
            sampled = rating_df.sample(n=10, replace=True)
            reviews_block = '\n---\n'.join(sampled['Review'].astype(str).tolist())

            if prompt_type == 'Basic':
                prompt = (
                    f'Based on these 10 reviews (all rated {rating}/10), '
                    'write one short representative review. No headers.\n\n'
                    f'Reviews:\n{reviews_block}'
                )
            else:
                persona = 'an angry critic' if rating <= 4 else 'a happy fan'
                prompt = (
                    f'You are {persona}. Write a short review based on these 10 reviews '
                    f'(all rated {rating}/10). No headers.\n\nReviews:\n{reviews_block}'
                )

            generated_text, status = call_gemini_smart(prompt)
            if status == 'DAILY_LIMIT':
                stop_execution = True
                break

            _, final_rating = predict_rating(generated_text)
            results.append(
                {
                    'Target_Rating': rating,
                    'Model': RESULT_MODEL_LABEL,
                    'Prompt_Type': prompt_type,
                    'Generated_Text': generated_text,
                    'DistilBERT_Rating': final_rating,
                    'Error': abs(rating - final_rating),
                }
            )

            pd.DataFrame(results).to_excel(OUTPUT_FILE, index=False)
            time.sleep(2)

results_df = pd.DataFrame(results)
if not results_df.empty:
    summary_df = (
        results_df.groupby(['Model', 'Prompt_Type', 'Target_Rating'])
        .agg(
            Mean_Error=('Error', 'mean'),
            Accuracy_Pct=('Error', lambda values: (values == 0).mean() * 100),
            Std_Deviation=('Error', 'std'),
            Sample_Count=('Generated_Text', 'count'),
        )
        .reset_index()
    )

    summary_df.to_excel(SUMMARY_FILE, index=False)
    results_df.to_excel(OUTPUT_FILE, index=False)

    print(f'\nRun paused or completed. Total samples in Excel: {len(results_df)}')
    print('\n--- Summary of performance so far ---')
    print(summary_df)
else:
    print('\nNo data found for processing.')
